# Tutorial: Inspect monitor specifications, geometry, and actions
This tutorial uses a monitor session to inspect task metadata, pre-event features, zonotope geometry, and action availability.

The Geometry15 policy sees only the current pre-event state. Specification text and channel names describe the task; recorded future inputs and exact references are evaluator-only data.

In [ ]:
from pzr import learning
import pandas as pd
import torch

torch.set_num_threads(1)
task = learning.load_task("robot_arm", bundle="starter", budget=40)

In [ ]:
print(task.metadata["spec"])
print("Inputs:", task.metadata["input_channels"])
print("Triggers:", task.metadata["trigger_labels"])

In [ ]:
from pzr.rtlola.scenarios import RtlolaTrace
trace = task.validation_traces.traces[0]
prefix = RtlolaTrace(trace.scenario, trace.trace_kind, trace.seed, trace.events[:20])
session = learning.MonitorSession(prefix, budget=task.budget)
for _ in range(5):
    transition = session.step(0)  # request girard
observation = session.observe(geometry=True)
print(transition)
pd.Series(dict(zip(task.dataset.feature_names, observation.features)))

## Interpret zonotope geometry
`center[d]` and `generators[d,m]` describe `center + generators @ xi`, with each component of `xi` between -1 and 1. Dynamic and total exports are separate; total includes constant calibration slack.

Signed coefficients and logical zero rows are retained. Do not assume row identities persist across steps, or that total generator columns can be matched to dynamic columns. The compact reducer dimension and exported logical dimension are distinct. The budget is the native transform bound, not a post-event dense-slot limit.

In [ ]:
print("Dynamic:", observation.dynamic.center.shape, observation.dynamic.generators.shape)
print("Total:", observation.total.center.shape, observation.total.generators.shape)
print("Counts:", observation.counts)
print("Known availability:", observation.action_mask)
print("Reduction needed:", observation.reduction_needed)

The arrays are read-only inspection data. Native transforms alone change the monitor. Custom encoders can use these exports now through `learning.FeatureEncoder` (see notebook 03). Recorded teacher bundles contain only Geometry15, so supervised GNN training requires geometry captured at the corresponding teacher states; learned reduction and forecasting require separate targets and soundness checks before any live use.